<a href="https://colab.research.google.com/github/marius-ne/CIE_ProjectB_Group13/blob/junchao/match%20nodes%20to%20bodies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIE 2025/26 RWTH, PROJECT B, GROUP 13
Junchao Yu, Marius Neuhalfen

ToDo:
- Find defective nodes by comparing perfect structure and imperfect structures for all scenarios
- Group defective nodes into regions (arc sections or track sections)
- Predict whether structure is perfect or imperfect
- Predict where the imperfection lies

There are 25.XXX for deformation and 24.XXX for stress

#find x-beam nodes' coordinate

In [ ]:
import pandas as pd
from google.colab import files

# 1. 读取您上传的两个文件
# 假设文件名分别为 'nodeExport.txt' 和 'x-beam.txt'
# 如果您的文件名不同，请修改这里
df_all = pd.read_csv('nodeExport.txt', sep='\t')
df_ids = pd.read_csv('x-beamnew.txt', sep='\t')

# 2. 清理列名（去除多余空格）并统一格式
# 提取坐标文件中的 ID, X, Y, Z (假设列名包含 "Node", "X", "Y", "Z")
df_all = df_all[['Node Number', 'X Location (m)', 'Y Location (m)', 'Z Location (m)']]
df_all.columns = ['Node', 'X', 'Y', 'Z']

# 提取 ID 文件中的 Node 列
df_ids = df_ids[['Node Number']]
df_ids.columns = ['Node']

# 3. 合并数据 (类似于 Excel 的 VLOOKUP)
#只保留 df_ids 中存在的节点
result = pd.merge(df_ids, df_all, on='Node', how='left')

# 4. 保存为 TXT 文件
output_filename = 'matched_nodes_coordinates.txt'
result.to_csv(output_filename, sep='\t', index=False)
print(f"文件已生成: {output_filename}")

# 5. 自动触发下载 (Colab 特有功能)
files.download(output_filename)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# 读取数据
df = pd.read_csv('x-beams withcoordinates new.txt', sep='\t')

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# 计算各轴的范围
x_range = df['X'].max() - df['X'].min()
y_range = df['Y'].max() - df['Y'].min()
z_range = df['Z'].max() - df['Z'].min()

# 设置绘图点，颜色设为粉色 (hotpink)
ax.scatter(df['X'], df['Y'], df['Z'], c='hotpink', marker='.', s=5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Point Cloud - True Scale (Pink)')

# 【关键步骤】设置坐标轴比例一致
# 这一步告诉 matplotlib 按照数据的真实比例来显示盒子
ax.set_box_aspect((x_range, y_range, z_range))

# 调整视角 (俯视)
ax.view_init(elev=90, azim=-90)

plt.savefig('3d_point_cloud_pink_true_scale.png')
plt.show()

#classify nodes to every body

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import matplotlib.cm as cm

# ================= 文件名配置 =================
# 请确保左侧文件栏已经上传了这两个文件，名字必须完全一致
coords_file = 'x-beams withcoordinates.txt'
map_file    = 'node_solid_map new.txt'
# ============================================

print("1. 正在读取文件...")

try:
    # --- 读取坐标文件 (Tab分隔) ---
    # 这里的 sep='\t' 很关键，因为 x-beams 文件是用 Tab 隔开的
    df_coords = pd.read_csv(coords_file, sep='\t')
    df_coords.columns = [c.strip() for c in df_coords.columns] # 去除列名空格
    print(f"   坐标文件读取成功: {len(df_coords)} 行")

    # --- 读取映射文件 (逗号分隔) ---
    df_map = pd.read_csv(map_file, sep=',', skipinitialspace=True)
    df_map.columns = [c.strip() for c in df_map.columns]
    print(f"   映射文件读取成功: {len(df_map)} 行")

    # --- 数据类型统一 ---
    # 强制把 Node ID 转为整数，防止匹配失败
    df_coords['Node'] = df_coords['Node'].astype(int)
    df_map['NodeID']  = df_map['NodeID'].astype(int)

    # --- 核心步骤：筛选与合并 ---
    # "inner" 模式表示只保留两个文件中都有的点
    # 这完美实现了您的需求："选出只包含 x-beams 中的点"
    df_merged = pd.merge(df_coords, df_map, left_on='Node', right_on='NodeID', how='inner')

    # 提取 BodyID
    body_col = 'BodyID' # 如果您的文件列名不一样，请在这里修改
    unique_bodies = df_merged[body_col].unique()
    # 去除背景 0 (如果存在)
    unique_bodies = [b for b in unique_bodies if b != 0]

    print(f"✅ 合并成功！")
    print(f"   共筛选出节点: {len(df_merged)}")
    print(f"   识别出独立部件: {len(unique_bodies)} 个")

    # --- 绘图 ---
    print("2. 正在生成 3D 彩色图像...")
    fig = plt.figure(figsize=(15, 12))
    ax = fig.add_subplot(111, projection='3d')

    # 生成颜色表 (使用 jet 色谱，对比度高)
    colors = cm.jet(np.linspace(0, 1, len(unique_bodies)))

    # 使用 groupby 加速绘图
    grouped = df_merged.groupby(body_col)

    for i, (body_id, group) in enumerate(grouped):
        if body_id == 0: continue

        # 画点
        ax.scatter(group['X'], group['Y'], group['Z'],
                   s=10,               # 点的大小
                   color=colors[i],    # 自动分配颜色
                   alpha=1.0,          # 不透明度
                   label=f'Body {int(body_id)}' if i < 10 else "") # 只显示前10个图例防止拥挤

    # 设置坐标轴标签
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'X-Beams Colored by BodyID ({len(unique_bodies)} Parts)')

    # 自动调整比例 (让模型看起来不变形)
    try:
        max_range = max(df_merged['X'].max()-df_merged['X'].min(),
                        df_merged['Y'].max()-df_merged['Y'].min(),
                        df_merged['Z'].max()-df_merged['Z'].min()) / 2.0
        mid_x = (df_merged['X'].max()+df_merged['X'].min()) * 0.5
        mid_y = (df_merged['Y'].max()+df_merged['Y'].min()) * 0.5
        mid_z = (df_merged['Z'].max()+df_merged['Z'].min()) * 0.5
        ax.set_xlim(mid_x - max_range, mid_x + max_range)
        ax.set_ylim(mid_y - max_range, mid_y + max_range)
        ax.set_zlim(mid_z - max_range, mid_z + max_range)
    except:
        pass

    plt.show()
    print("🎉 绘图完成！")

except Exception as e:
    print(f"❌ 发生错误: {e}")
    # 如果报错找不到列，打印出来看看
    import traceback
    traceback.print_exc()

In [ ]:
import pandas as pd
import plotly.express as px

# ================= 文件名配置 =================
coords_file = 'x-beams withcoordinates.txt'
map_file    = 'node_solid_map new.txt'
# ============================================

try:
    print("1. 读取并合并数据...")
    # 读取
    df_coords = pd.read_csv(coords_file, sep='\t')
    df_coords.columns = [c.strip() for c in df_coords.columns]

    df_map = pd.read_csv(map_file, sep=',', skipinitialspace=True)
    df_map.columns = [c.strip() for c in df_map.columns]

    # 转换类型
    df_coords['Node'] = df_coords['Node'].astype(int)
    df_map['NodeID']  = df_map['NodeID'].astype(int)

    # 合并
    df_merged = pd.merge(df_coords, df_map, left_on='Node', right_on='NodeID', how='inner')

    # 过滤掉 BodyID=0 (背景)
    if 'BodyID' in df_merged.columns:
        df_merged = df_merged[df_merged['BodyID'] != 0]
        # 把 BodyID 变成字符串，这样 Plotly 才会把它当成离散的颜色分类，而不是连续的渐变色
        df_merged['BodyID'] = df_merged['BodyID'].astype(str)

    print(f"✅ 准备绘图 (共 {len(df_merged)} 个点)...")

    # --- 使用 Plotly 进行交互式绘图 ---
    fig = px.scatter_3d(df_merged,
                        x='X',
                        y='Y',
                        z='Z',
                        color='BodyID',   # 根据部件分色
                        opacity=1.0,      # 不透明
                        title="Interactive X-Beam Visualization (Drag to Rotate)",
                        width=1000,
                        height=800)

    # 优化点的显示：让点稍微小一点，看着精致
    fig.update_traces(marker=dict(size=3))

    # 隐藏过多的图例 (如果有几百个部件，图例会挡住图)
    fig.update_layout(showlegend=False)

    # 设置比例固定 (防止模型变形)
    fig.update_layout(scene=dict(aspectmode='data'))

    print("🎉 图表已生成！请使用鼠标拖拽旋转查看。")
    fig.show()

except Exception as e:
    print(f"❌ 出错: {e}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ================= 1. 读取与处理数据 =================
# 读取坐标文件
df_coords = pd.read_csv('x-beams withcoordinates new.txt', sep='\t')
df_coords.columns = [c.strip() for c in df_coords.columns]

# 读取映射文件
df_map = pd.read_csv('node_solid_map new.txt', sep=',', skipinitialspace=True)
df_map.columns = [c.strip() for c in df_map.columns]

# 统一数据类型
df_coords['Node'] = df_coords['Node'].astype(int)
df_map['NodeID'] = df_map['NodeID'].astype(int)

# 合并两个表格
df_merged = pd.merge(df_coords, df_map, left_on='Node', right_on='NodeID', how='inner')

# 按 BodyID 排序
df_merged = df_merged.sort_values(by='BodyID')

# 选取需要的列
output_df = df_merged[['BodyID', 'Node', 'X', 'Y', 'Z']]

# 保存为新 TXT 文件
output_df.to_csv('merged_body_nodes.txt', sep='\t', index=False)
print("✅ 文件已生成: merged_body_nodes.txt")

# ================= 2. 定义可视化函数 =================
def visualize_body(target_body_id):
    """
    输入: target_body_id (整数或浮点数)
    功能: 画出所有背景点(灰色)和指定BodyID的点(红色)
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # 1. 画背景 (使用原始坐标文件中的所有点)
    # alpha=0.05 让背景非常淡，不抢眼
    ax.scatter(df_coords['X'], df_coords['Y'], df_coords['Z'],
               c='grey', s=1, alpha=0.5, label='Background')

    # 2. 画目标 Body
    target_data = df_merged[df_merged['BodyID'] == target_body_id]

    if not target_data.empty:
        ax.scatter(target_data['X'], target_data['Y'], target_data['Z'],
                   c='red', s=20, alpha=1.0, label=f'Body {int(target_body_id)}')

        # 自动调整视角聚焦到目标物体
        mid_x = target_data['X'].mean()
        mid_y = target_data['Y'].mean()
        mid_z = target_data['Z'].mean()
        # 这里只是简单的聚焦，您可以手动旋转
        # ax.set_xlim(mid_x - 0.05, mid_x + 0.05)

        print(f"📍 已找到 Body {target_body_id}，包含 {len(target_data)} 个节点。")
    else:
        print(f"⚠️ 未找到 BodyID 为 {target_body_id} 的数据！请检查ID是否正确。")

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'Highlight: Body {target_body_id}')
    ax.legend()
    plt.show()

# ================= 3. 使用示例 =================
# 获取一个存在的 ID 来测试
sample_id = df_merged['BodyID'].values[0]

# 调用函数 (您可以把 sample_id 换成您想看的任何数字)
visualize_body(-340693)

#match x-beam bodys' names

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# ================= 配置 =================
# 1. APDL 导出的节点数据 (之前的 merged_body_nodes.txt)
#    如果您还没有这个文件，请用上一轮的脚本生成
data_file = 'xbeamID_nodes.txt'

# 2. Workbench 导出的名字映射 (刚刚生成的 wb_name_map.txt)
name_file = 'wb_name_map new.txt'

# ================= 1. 读取数据 =================
print("1. 读取并计算 ID 中心...")
# 读取节点数据
df_nodes = pd.read_csv(data_file, sep='\t')
df_nodes.columns = [c.strip() for c in df_nodes.columns]

# --- 核心步骤：计算每个 BodyID 的几何中心 ---
# 我们不需要知道 ID 是怎么生成的，算出它的中心在哪里就行
df_centers = df_nodes.groupby('BodyID')[['X', 'Y', 'Z']].mean().reset_index()
# 重命名，方便区分
df_centers.columns = ['BodyID', 'ID_X', 'ID_Y', 'ID_Z']

print(f"   已计算 {len(df_centers)} 个内部 ID 的中心坐标。")

# 读取名字映射文件
try:
    df_names = pd.read_csv(name_file, sep=',')
    print(f"2. 读取名字映射表... 成功加载 {len(df_names)} 个部件名。")
except:
    print("⚠️ 警告：找不到 wb_name_map.txt。无法显示名字，只能显示 ID。")
    df_names = pd.DataFrame()

# ================= 2. 匹配算法 (最近邻搜索) =================
# 创建一个字典： BodyID -> Name
id_to_name_map = {}

if not df_names.empty:
    print("3. 正在进行几何匹配 (Matching Geometry)...")
    from scipy.spatial import cKDTree

    # 构建 ID 中心点的 KD-Tree (快速搜索)
    id_points = df_centers[['ID_X', 'ID_Y', 'ID_Z']].values
    tree = cKDTree(id_points)

    # 遍历每一个 Workbench 的名字，去寻找最近的 ID
    wb_points = df_names[['Centroid_X', 'Centroid_Y', 'Centroid_Z']].values

    # 查询最近点 (返回 距离 和 索引)
    distances, indices = tree.query(wb_points)

    # 建立映射
    for i, idx in enumerate(indices):
        dist = distances[i]
        # 如果距离非常小（比如小于 1mm），说明匹配成功
        # 注意单位！如果您模型很大，这个阈值可能要调整
        if dist < 0.01:
            matched_id = df_centers.iloc[idx]['BodyID']
            part_name  = df_names.iloc[i]['Name']
            id_to_name_map[matched_id] = part_name
        else:
            print(f"   ⚠️ 未匹配到: {df_names.iloc[i]['Name']} (最近距离 {dist:.4f})")

print(f"   匹配完成！共关联了 {len(id_to_name_map)} 个部件。")

# ================= 3. 增强版可视化函数 =================
def find_and_plot(query):
    """
    query: 可以是 BodyID (数字)，也可以是名字 (字符串)
    """
    target_id = None
    target_name = "Unknown"

    # 判断用户输入的是名字还是ID
    if isinstance(query, str):
        # 此时 query 是名字，反向查找 ID
        # 这是一个简单的反向搜索
        for bid, name in id_to_name_map.items():
            if query in name: # 支持模糊搜索
                target_id = bid
                target_name = name
                break
        if target_id is None:
            print(f"❌ 找不到包含 '{query}' 的部件名字。")
            return
    else:
        # 此时 query 是数字 ID
        target_id = query
        target_name = id_to_name_map.get(target_id, "Unnamed Part")

    # 开始绘图
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # 画背景 (灰色)
    ax.scatter(df_nodes['X'], df_nodes['Y'], df_nodes['Z'],
               c='lightgrey', s=1, alpha=0.5)

    # 画目标 (红色)
    target_data = df_nodes[df_nodes['BodyID'] == target_id]

    if not target_data.empty:
        ax.scatter(target_data['X'], target_data['Y'], target_data['Z'],
                   c='red', s=20, alpha=1.0, label=f'{target_name}\n(ID: {int(target_id)})')

        # 自动聚焦
        mid_x, mid_y, mid_z = target_data[['X','Y','Z']].mean()
        range_ = 0.03 # 视口范围
        ax.set_xlim(mid_x - range_, mid_x + range_)
        ax.set_ylim(mid_y - range_, mid_y + range_)
        ax.set_zlim(mid_z - range_, mid_z + range_)

        print(f"📍 已定位部件: {target_name} (ID: {int(target_id)})")
    else:
        print(f"❌ 数据中找不到 ID {target_id}")

    ax.legend()
    plt.title(f"Visualizing: {target_name}")
    plt.show()

# ================= 4. 测试 =================
# 示例：如果您有一个部件叫 "Beam_Front"，您可以直接搜名字
# find_and_plot("Beam_Front")

# 或者搜之前看到的 ID
sample_id = df_centers.iloc[0]['BodyID']
find_and_plot("Body_40")

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# ================= 文件配置 =================
node_file = 'merged_body_nodes.txt'  # 包含 BodyID, Node, X, Y, Z
name_file = 'wb_name_map new.txt'      # 包含 Name, Centroid_X, ...
output_file = 'final_named_structure.txt'
# ===========================================

print("1. 读取文件...")
try:
    # 读取节点数据
    df_nodes = pd.read_csv(node_file, sep='\t')
    df_nodes.columns = [c.strip() for c in df_nodes.columns]

    # 读取名字映射
    df_names = pd.read_csv(name_file, sep=',')
    df_names.columns = [c.strip() for c in df_names.columns]

    print(f"   节点数据: {len(df_nodes)} 行")
    print(f"   名字数据: {len(df_names)} 个部件")

except Exception as e:
    print(f"❌ 读取失败: {e}")
    print("   请确保 merged_body_nodes.txt 和 wb_name_map.txt 都在当前文件夹中！")
    exit()

print("2. 计算几何中心并匹配...")

# --- 步骤 A: 计算 APDL 数据的中心 ---
# 根据 BodyID 分组，计算每组的平均坐标 (X, Y, Z)
df_centers = df_nodes.groupby('BodyID')[['X', 'Y', 'Z']].mean().reset_index()
apdl_points = df_centers[['X', 'Y', 'Z']].values

# --- 步骤 B: 准备 Workbench 数据的中心 ---
wb_points = df_names[['Centroid_X', 'Centroid_Y', 'Centroid_Z']].values
wb_names = df_names['Name'].values

# --- 步骤 C: 建立 KD-Tree 进行最近邻搜索 ---
# 我们拿着 Workbench 的中心，去 APDL 的中心里找“最近的那个”
tree = cKDTree(apdl_points)

# query(查询点, k=1) 返回最近距离(dists)和对应的索引(idxs)
dists, idxs = tree.query(wb_points)

# --- 步骤 D: 建立映射字典 {BodyID : Name} ---
id_name_map = {}
matched_count = 0

# 设定一个容差 (例如 0.001 即 1mm，如果单位是m的话)
# 如果中心点距离小于这个值，认为是同一个物体
tolerance = 0.001

for i, idx in enumerate(idxs):
    dist = dists[i]
    if dist < tolerance:
        # 找到了！
        found_body_id = df_centers.iloc[idx]['BodyID']
        found_name = wb_names[i]
        id_name_map[found_body_id] = found_name
        matched_count += 1
    else:
        print(f"   ⚠️ 未匹配警告: {wb_names[i]} 距离最近的网格中心太远 ({dist:.4f})")

print(f"✅ 匹配完成！成功关联了 {matched_count} / {len(df_names)} 个部件。")

# ================= 3. 生成最终表格 =================
print("3. 生成导出数据...")

# 将名字映射回原始节点表
# map 函数会自动根据 BodyID 填入对应的 Name
df_nodes['BodyName'] = df_nodes['BodyID'].map(id_name_map)

# 过滤掉没匹配上的行 (dropna)
df_final = df_nodes.dropna(subset=['BodyName'])

# 按照 Name 排序，如果 Name 相同则按 Node 排序
df_final = df_final.sort_values(by=['BodyName', 'Node'])

# 整理列顺序
# 格式: Name, Node, X, Y, Z
output_df = df_final[['BodyName', 'Node', 'X', 'Y', 'Z']]

# 导出
output_df.to_csv(output_file, sep='\t', index=False)

print(f"🎉 文件已保存为: {output_file}")
print("-" * 30)
print("文件预览 (前 10 行):")
print(output_df.head(10))

#FINAL result

In [ ]:
import pandas as pd
import plotly.express as px

# ================= 文件名 =================
filename = "final_named_structure.txt"
# ========================================

try:
    print("1. 读取数据...")
    df = pd.read_csv(filename, sep='\t')
    # 清理列名空格
    df.columns = [c.strip() for c in df.columns]

    print(f"   成功读取 {len(df)} 个节点。")
    print(f"   包含部件数量: {len(df['BodyName'].unique())}")

    print("2. 生成交互式图表...")

    # 使用 Plotly Express 创建 3D 散点图
    fig = px.scatter_3d(
        df,
        x='X',
        y='Y',
        z='Z',
        color='BodyName',          # 根据名字自动着色
        hover_name='BodyName',     # 鼠标悬停时的大标题
        hover_data={               # 定制悬停显示的详细信息
            'BodyName': False,     # 标题已显示，这里不重复
            'Node': True,          # 显示节点号
            'X': True,             # 显示坐标
            'Y': True,
            'Z': True
        },
        title='Interactive Structure Visualization (Hover to see Name)',
        opacity=1.0,               # 点的不透明度
        width=1200,                # 图表宽度
        height=900                 # 图表高度
    )

    # 优化显示效果
    fig.update_traces(marker=dict(size=4)) # 把点稍微调大一点，更好选中

    # 如果部件太多（超过20个），隐藏图例防止挡住画面
    if len(df['BodyName'].unique()) > 20:
        fig.update_layout(showlegend=False)
        print("   (提示：部件较多，已自动隐藏图例以保持视野清晰)")

    # 锁定比例，防止模型被压扁
    fig.update_layout(scene=dict(aspectmode='data'))

    # 鼠标悬停样式优化
    fig.update_layout(hoverlabel=dict(
        bgcolor="white",
        font_size=14,
        font_family="Rockwell"
    ))

    print("🎉 图表已生成！请在下方查看。")
    fig.show()

except Exception as e:
    print(f"❌ 发生错误: {e}")